# 02 — Concat embeddings (ablation study)

Sometimes a single concatenated embedding outperforms stacking the same components as separate base learners. This notebook shows how to use `build_concat_h5ad` to generate pairwise + all-N concatenations as scBoost inputs.

In [ ]:
import numpy as np
from pathlib import Path
from scboost_ensemble import build_concat_h5ad, StackingEnsembleDonorMetaCached

## Build a synthetic .npz with multiple latent matrices

In practice this comes from your annotator (e.g. popV saves latents this way). Here we create one for illustration.

In [ ]:
import anndata as ad, pandas as pd
import scipy.sparse as sp
rng = np.random.default_rng(0); n = 200
cell_ids = np.array([f'c{i:04d}' for i in range(n)])
# Reference h5ad needed for obs metadata
ref = ad.AnnData(X=sp.csr_matrix((n, 1), dtype=np.float32),
                  obs=pd.DataFrame({'cell_type': np.random.choice(['A','B','C'], n),
                                     'batch':    np.random.choice([f'd{i}' for i in range(5)], n)},
                                    index=cell_ids))
ref.write_h5ad('/tmp/scb_ref.h5ad')
# Save 4 fake latents into one .npz
np.savez_compressed('/tmp/scb_latents.npz', cell_ids=cell_ids,
                     scVI=rng.normal(size=(n, 8)),
                     scANVI=rng.normal(size=(n, 8)),
                     Harmony=rng.normal(size=(n, 10)),
                     Scanorama=rng.normal(size=(n, 10)))
print('synthetic inputs written to /tmp/')

## Build h5ad with concat embeddings

In [ ]:
adata = build_concat_h5ad(
    npz_path='/tmp/scb_latents.npz',
    reference_h5ad='/tmp/scb_ref.h5ad',
    out='/tmp/scb_concat.h5ad',
)
print('obsm keys:', list(adata.obsm.keys()))

With 4 base latents, we get C(4,2)=6 pairwise concats + 1 all-4 concat = 7 entries.

## Run scBoost on the concat embeddings
Choose either (a) one big run stacking all concats or (b) one run per concat.

In [ ]:
# Option: stack all pairwise concats together
pair_keys = [k for k in adata.obsm if k.startswith('concat_') and k != 'concat_all']
stacker = StackingEnsembleDonorMetaCached(
    obsm_keys=pair_keys, cv_scheme='donor', n_folds=4,
    early_stopping_rounds=5, device='cpu', use_batch_feature=False,
)
adata.obs['split'] = 'train'  # synthetic data has no holdout in this demo
stacker.fit(adata)
print('method importance:', stacker.get_method_importance())